# Train the priority classifier (v2.0) on Colab

DistilBERT (`distilbert-base-uncased`) fine-tuned on `insanar/prior-mail-priority` (config `v2`).

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- This notebook clones the repo from GitHub; the migration lives on `main`.

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

2.11.0+cu128 Tesla T4


## 2. Clone the repo

Private org repo → use a GitHub PAT with `repo` scope.

In [ ]:
!git clone https://github.com/PJK-GM095-PIJAK/prior-mail-model.git
%cd prior-mail-model
# !git checkout <branch-or-sha>   # optional: pin a specific commit

Cloning into 'prior-mail-model'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 195 (delta 89), reused 152 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (195/195), 98.12 KiB | 1.58 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/prior-mail-model/prior-mail-model


## 3. Install dependencies

Not `pip install -e .` — Colab's Python doesn't satisfy the `>=3.11,<3.12` pin. Install the libs directly and put the repo root on `PYTHONPATH`. (`sentencepiece` is no longer needed — DistilBERT uses WordPiece.)

In [ ]:
!pip install -q transformers datasets accelerate evaluate wandb emoji huggingface_hub
%env PYTHONPATH=.

env: PYTHONPATH=.


## 4. Experiment tracking (pick ONE)

Live tracking with W&B (project `priormail`):

In [ ]:
import wandb
wandb.login()   # paste your W&B key

True

...or skip tracking. If so, run **only** the line below in its own cell — with **no trailing comment** (a comment silently breaks `%env`):

In [ ]:
%env WANDB_MODE=offline

env: WANDB_MODE=offline


## 5. Build the dataset

Downloads `insanar/prior-mail-priority:v2` (public) and writes `data/processed/priority`.

In [ ]:
!make data

python -m src.data.prepare
2026-06-09 00:57:51,164 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-06-09 00:57:51,593 INFO datasets: TensorFlow version 2.20.0 available.
2026-06-09 00:57:51,594 INFO datasets: JAX version 0.7.2 available.
2026-06-09 00:57:52,205 INFO httpx: HTTP Request: HEAD https://huggingface.co/datasets/insanar/prior-mail-priority/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-09 00:57:52,206 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-09 00:57:52,246 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/insanar/prior-mail-priority/4aae4bd09e54cf2f7f4acfa9120c854990ebf36f/README.md "HTTP/1.1 200 OK"
2026-06-09 00:57:52,330 INFO httpx: HTTP Request: HEAD https://huggingface.co/datasets/insanar/prior-mail-priority/resolve/4aae4bd09e54cf2f7f4acfa9120c854990ebf36f/prior-mail-prio

## 6. Train

DistilBERT on ~4.3k examples × 4 epochs ≈ a few minutes on a T4. The config requests `bf16`; on a T4 (no bf16) the trainer auto-falls back to `fp16`. Class weights (urgent/high ≈ 3.4×) apply automatically via `class_weights: balanced`.

In [ ]:
!make train config=configs/priority_v2.yaml

python -m src.training.train_priority --config configs/priority_v2.yaml
2026-06-09 00:57:56,572 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-06-09 00:57:57,806 INFO datasets: TensorFlow version 2.20.0 available.
2026-06-09 00:57:57,809 INFO datasets: JAX version 0.7.2 available.
[INFO] Running in WANDB offline mode
2026-06-09 00:58:05,415 INFO src.utils.seeding: Global seed set to 42 (deterministic=True)
2026-06-09 00:58:05,422 INFO __main__: Starting priority training | model=distilbert-base-uncased seed=42 git=17ac82b
2026-06-09 00:58:05,611 INFO httpx: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-09 00:58:05,705 INFO httpx: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-09 00:58:05,791 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=fal

## 7. Evaluate against the gates

Promotion gates: macro-F1 ≥ 0.80, per-class recall ≥ 0.65, p95 < 500 ms on CPU.

In [ ]:
!make eval config=configs/priority_v2.yaml

python -m src.eval.eval_priority --config configs/priority_v2.yaml
2026-06-09 01:17:20,779 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-06-09 01:17:21,229 INFO datasets: TensorFlow version 2.20.0 available.
2026-06-09 01:17:21,230 INFO datasets: JAX version 0.7.2 available.
Loading weights: 100% 104/104 [00:00<00:00, 9732.22it/s]
2026-06-09 01:18:21,039 INFO src.eval.benchmarks: Latency over 200 inputs: {'mean_ms': 70.21037540499492, 'p50_ms': 65.41316350012494, 'p95_ms': 96.61467090027143, 'n': 200.0}
2026-06-09 01:18:21,040 INFO __main__: Gates PASSED | macro_f1=0.972 p95=97ms -> {'macro_f1': True, 'per_class_recall': True, 'latency_p95': True}


In [ ]:
# ── Step 6b: write model card (required before export) ────────────────────────
import json, os

with open("eval/results/priority/eval_report.json") as f:
    m = json.load(f)["metrics"]

lines = [
    "# Priority Classifier v2.0",
    "",
    "DistilBERT (`distilbert-base-uncased`) fine-tuned on `insanar/prior-mail-priority` (config `v2`).",
    "",
    "## Performance (held-out test set)",
    "",
    "| Metric | Value |",
    "|---|---|",
    f"| Macro F1 | {m['macro_f1']:.3f} |",
    f"| Recall — urgent | {m['recall_urgent']:.3f} |",
    f"| Recall — high | {m['recall_high']:.3f} |",
    f"| Recall — normal | {m['recall_normal']:.3f} |",
    f"| Recall — low | {m['recall_low']:.3f} |",
    f"| Inference p95 (CPU) | {m['p95_ms']:.0f} ms |",
    "",
    "## Known limits",
    "- Topic→priority proxy, not true urgency understanding",
    "- Trained on English only",
    "- Dataset includes synthetic examples",
]

os.makedirs("checkpoints/priority_v2", exist_ok=True)
with open("checkpoints/priority_v2/model_card.md", "w") as f:
    f.write("\n".join(lines))

print("✓ model_card.md written")


✓ model_card.md written


## 8. Save the checkpoint

Zip + download (Drive is flaky):

In [ ]:
!cd checkpoints && zip -qr priority_v2.zip priority_v2
from google.colab import files
files.download("checkpoints/priority_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Or publish to the HuggingFace Hub (needs a **write** token; the uploader handles the Xet stall itself):

In [ ]:
!cp eval/results/priority/eval_report.json checkpoints/priority_v2/

In [ ]:
from huggingface_hub import logout
logout()


In [ ]:
from huggingface_hub import login
login()   # HF token with write access

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("HuggingFace WRITE token: ")


HuggingFace WRITE token: ··········


In [ ]:
!python -m src.exporter.export --checkpoint checkpoints/priority_v2 --hf-org insanar --version v2.0

2026-06-09 01:33:40,081 INFO __main__: packaged config.json
2026-06-09 01:33:40,082 INFO __main__: packaged tokenizer.json
2026-06-09 01:33:40,083 INFO __main__: packaged tokenizer_config.json
2026-06-09 01:33:40,083 INFO __main__: packaged training_config.yaml
2026-06-09 01:33:40,083 INFO __main__: packaged eval_report.json
2026-06-09 01:33:40,084 INFO __main__: packaged model_card.md
2026-06-09 01:33:40,459 INFO __main__: packaged model.safetensors
2026-06-09 01:33:40,459 INFO __main__: Packaged 7 artifacts into checkpoints/priority_v2/dist
2026-06-09 01:33:41,080 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/insanar/priormail-priority/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-06-09 01:33:41,176 INFO httpx: HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 409 Conflict"
2026-06-09 01:33:42,254 INFO httpx: HTTP Request: POST https://huggingface.co/api/models/insanar/priormail-priority/preupload/main "HTTP/1.1 200 OK"
2026-06-09 0